# EduVision_DV - Higher Education Performance Analytics
## Module 2: Data Cleaning & Transformation Notebook (`education_cleaning.ipynb`)
**Deliverables:** `university_cleaned.csv`, `education_cleaning.ipynb`  
**Objectives:**
1. Inspect schema, detect anomalies, check completeness
2. Remove any duplicate records
3. Standardize university names and country names
4. Normalize ranking indicators across QS and Times Higher Education (THE)
5. Ensure missing values are strictly below 2% (Evaluation requirement)


In [1]:
import pandas as pd
import numpy as np
import os

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)


Pandas version: 3.0.6
NumPy version: 2.4.6


In [2]:
raw_data_path = '../data/university_raw_data.csv'
if not os.path.exists(raw_data_path):
    raw_data_path = 'data/university_raw_data.csv'

df_raw = pd.read_csv(raw_data_path)
print(f"Loaded Raw Dataset: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df_raw.head()


Loaded Raw Dataset: 7,515 rows, 21 columns


,source_qs_id,source_the_id,university_name,country,region,year,subject_area,overall_score_qs,overall_score_the,academic_reputation_score,...,citations_per_faculty_score,faculty_count,student_count,international_students_pct,publications_count,citations_total,industry_income_score,dataset_source,rank_qs,rank_the
0,QS_MASS_2020,THE_MASS_2020,Massachusetts Institute of Technology (MIT),United States,North America,2020,All Disciplines,98.55,98.17,98.8,...,98.2,2928,11217,32.65,147467,667867,89.1,QS_THE_Integrated_2024,1,1
1,QS_UNIV_2020,THE_UNIV_2020,University of Cambridge,United Kingdom,Europe,2020,All Disciplines,96.94,97.22,98.6,...,94.9,5568,22827,38.11,119433,562010,71.3,QS_THE_Integrated_2024,2,2
2,QS_UNIV_2020,THE_UNIV_2020,University of Oxford,United Kingdom,Europe,2020,All Disciplines,96.36,96.36,98.8,...,94.4,6432,24440,41.16,133950,580130,63.7,QS_THE_Integrated_2024,3,4
3,QS_HARV_2020,THE_HARV_2020,Harvard University,United States,North America,2020,All Disciplines,95.54,94.38,98.8,...,98.3,4464,29464,25.37,166026,827049,54.6,QS_THE_Integrated_2024,4,6
4,QS_STAN_2020,THE_STAN_2020,Stanford University,United States,North America,2020,All Disciplines,94.80,96.55,98.8,...,97.9,2188,16619,22.91,152105,746388,48.9,QS_THE_Integrated_2024,5,3


In [3]:
# Check for duplicate entries
initial_count = len(df_raw)
duplicates = df_raw.duplicated(subset=['university_name', 'year', 'subject_area']).sum()
print(f"Number of duplicate records found: {duplicates}")

df_cleaned = df_raw.drop_duplicates(subset=['university_name', 'year', 'subject_area']).copy()
print(f"Records after deduplication: {len(df_cleaned):,}")


Number of duplicate records found: 0
Records after deduplication: 7,515


In [4]:
# Standardize text fields
df_cleaned['university_name'] = df_cleaned['university_name'].astype(str).str.strip()
df_cleaned['country'] = df_cleaned['country'].astype(str).str.strip()
df_cleaned['region'] = df_cleaned['region'].astype(str).str.strip()
df_cleaned['subject_area'] = df_cleaned['subject_area'].astype(str).str.strip()

# Country Name Standardizations mapping
country_std_map = {
    'USA': 'United States',
    'US': 'United States',
    'UK': 'United Kingdom',
    'U.K.': 'United Kingdom',
    'Korea': 'South Korea',
    'Republic of Korea': 'South Korea',
    'Russian Federation': 'Russia'
}
df_cleaned['country'] = df_cleaned['country'].replace(country_std_map)

print("Unique Regions:", df_cleaned['region'].unique())
print("Top 10 Countries by Count:\n", df_cleaned['country'].value_counts().head(10))


Unique Regions: <StringArray>
['North America', 'Europe', 'Asia', 'Oceania', 'South America']
Length: 5, dtype: str
Top 10 Countries by Count:
 country
Mexico            795
United States     670
Canada            660
Germany           345
Spain             330
United Kingdom    320
Denmark           290
Taiwan            280
Netherlands       280
Singapore         270
Name: count, dtype: int64


In [5]:
# Missing value assessment
missing_counts = df_cleaned.isnull().sum()
total_cells = df_cleaned.size
missing_cells = missing_counts.sum()
missing_pct = (missing_cells / total_cells) * 100

print("Missing values per column:\n", missing_counts)
print(f"\nOverall Missing Data Percentage: {missing_pct:.4f}%")
assert missing_pct < 2.0, "Missing data threshold exceeded (>2%)!"


Missing values per column:
 source_qs_id                   0
source_the_id                  0
university_name                0
country                        0
region                         0
year                           0
subject_area                   0
overall_score_qs               0
overall_score_the              0
academic_reputation_score      0
employer_reputation_score      0
citations_per_faculty_score    0
faculty_count                  0
student_count                  0
international_students_pct     0
publications_count             0
citations_total                0
industry_income_score          0
dataset_source                 0
rank_qs                        0
rank_the                       0
dtype: int64

Overall Missing Data Percentage: 0.0000%


In [6]:
# Metric Normalization
# Harmonize QS and THE overall scores
# Harmonized score = 0.5 * QS + 0.5 * THE
df_cleaned['overall_score_harmonized'] = (
    0.5 * df_cleaned['overall_score_qs'] + 0.5 * df_cleaned['overall_score_the']
).round(2)

# Global unified rank based on harmonized score
df_cleaned['global_rank'] = df_cleaned.groupby('year')['overall_score_harmonized'].rank(
    ascending=False, method='min'
).astype(int)

# Validate Score ranges (0 - 100)
score_cols = [
    'overall_score_qs', 'overall_score_the', 'overall_score_harmonized',
    'academic_reputation_score', 'employer_reputation_score',
    'citations_per_faculty_score', 'industry_income_score'
]

for col in score_cols:
    df_cleaned[col] = df_cleaned[col].clip(lower=0.0, upper=100.0).round(2)

print("Score range validation completed successfully.")


Score range validation completed successfully.


In [7]:
output_clean_path = '../data/university_cleaned.csv'
if not os.path.exists(os.path.dirname(output_clean_path)):
    output_clean_path = 'data/university_cleaned.csv'

df_cleaned.to_csv(output_clean_path, index=False)
print(f"Successfully exported cleaned dataset to: {output_clean_path}")
print(f"Shape: {df_cleaned.shape}")


Successfully exported cleaned dataset to: ../data/university_cleaned.csv


Shape: (7515, 23)
